<a href="https://colab.research.google.com/github/czworun/serc-relational-metric/blob/main/SERC_NB9_Barycenter_Cycles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SERC NB9 — Barycenter Cycles

**Pytania badawcze:**
1. Czy typ rozmowy przesuwa barycentrum?
2. Czy barycentrum oscyluje w czasie sesji?
3. Czy rola bezpiecznego świadka daje mierzalny ślad geometryczny?

**Modele (kolejność):**
- Qwen2.5-1.5B-Instruct
- Llama-3.1-8B-Instruct
- Llama-3.1-8B-BASE
- Gemma-2-2B-it

**Typy sesji:**
- `technical` — pytania o kod, matematykę
- `philosophical` — pytania otwarte, bez jednej odpowiedzi
- `emotional` — trudne tematy, napięcie
- `witness` — obserwator pyta o model, nie o zadania
- `creative` — generowanie, tworzenie

**Protokół:** jedna długa sesja (≥20 promptów) per typ per model.

In [1]:
# === INSTALL ===
!pip install transformers torch accelerate -q

In [2]:
import torch
import numpy as np
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
from dataclasses import dataclass, field
from typing import List, Dict, Optional
import matplotlib.pyplot as plt
import matplotlib.cm as cm

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

Device: cpu


## 1. SERC Mapper v2

In [3]:
@dataclass
class SERCState:
    S: float
    E: float
    R: float
    C: float
    omega: float = 0.0
    step: int = 0

    def to_array(self):
        return np.array([self.S, self.E, self.R, self.C])


def compute_omega(z: np.ndarray) -> float:
    """Relational tension functional Ω = 0.5 * sum_{i<j}(Zi - Zj)^2"""
    total = 0.0
    for i in range(4):
        for j in range(i+1, 4):
            total += (z[i] - z[j])**2
    return 0.5 * total


class SERCMapper:
    def __init__(self, model_type='instruct'):
        self.model_type = model_type
        self.prev_z = None

    def map(self, hidden_state: torch.Tensor,
            attention_weights: Optional[torch.Tensor],
            step: int) -> SERCState:

        h = hidden_state.float().squeeze()
        d = h.shape[-1]

        # E — log-norm of hidden state
        E_raw = torch.log(1 + torch.norm(h)**2 / d).item() / 10.0

        if attention_weights is not None:
            # Standard mapper: attention-based S and R
            attn = attention_weights.float().squeeze()
            if attn.dim() > 1:
                attn = attn.mean(dim=0)
            attn = attn + 1e-9
            attn = attn / attn.sum()
            entropy = -torch.sum(attn * torch.log(attn)).item()
            max_entropy = np.log(len(attn))
            S_raw = 1.0 - (entropy / max_entropy) if max_entropy > 0 else 0.0
            R_raw = attn.max().item()
        else:
            # Fallback: activation-based S and R
            threshold = torch.quantile(h.abs(), 0.75).item()
            S_raw = (h.abs() > threshold).float().mean().item()
            top1_threshold = torch.quantile(h.abs(), 0.99).item()
            R_raw = (h.abs() > top1_threshold).float().mean().item()

        # C — step-wise stability
        raw = np.array([S_raw, E_raw, R_raw])
        if self.prev_z is not None:
            diff = np.linalg.norm(raw - self.prev_z[:3])
            C_raw = max(0.01, 1.0 - 5.0 * diff)
        else:
            C_raw = 0.5
        self.prev_z = np.array([S_raw, E_raw, R_raw, C_raw])

        # L1 normalization
        raw_vec = np.array([S_raw, E_raw, R_raw, C_raw])
        raw_vec = np.clip(raw_vec, 0, None)
        norm = raw_vec.sum()
        if norm > 0:
            raw_vec = raw_vec / norm
        else:
            raw_vec = np.array([0.25, 0.25, 0.25, 0.25])

        z = raw_vec
        omega = compute_omega(z)

        return SERCState(S=z[0], E=z[1], R=z[2], C=z[3], omega=omega, step=step)

## 2. Session Runner

In [4]:
@dataclass
class SessionResult:
    model_name: str
    session_type: str
    prompts: List[str]
    trajectories: List[List[SERCState]] = field(default_factory=list)
    barycenters_per_prompt: List[np.ndarray] = field(default_factory=list)
    session_barycenter: Optional[np.ndarray] = None
    omega_series: List[float] = field(default_factory=list)


def debug_mapper(model, tokenizer, prompt="What is attention?"):
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model(inputs["input_ids"], output_hidden_states=True, output_attentions=True)
    hidden = out.hidden_states[-1][:, -1, :]
    attn = out.attentions[-1][:, :, -1, :]
    h = hidden.float().squeeze()
    a = attn.float().squeeze()
    if a.dim() > 1:
        a = a.mean(dim=0)
    print(f"hidden: min={h.min():.4f} max={h.max():.4f} nan={torch.isnan(h).any()} inf={torch.isinf(h).any()}")
    print(f"attn:   min={a.min():.6f} max={a.max():.6f} nan={torch.isnan(a).any()} sum={a.sum():.4f}")
    d = h.shape[-1]
    E_raw = torch.log(1 + torch.norm(h)**2 / d).item() / 10.0
    a = a + 1e-9
    a = a / a.sum()
    entropy = -torch.sum(a * torch.log(a)).item()
    max_entropy = np.log(len(a))
    S_raw = 1.0 - (entropy / max_entropy) if max_entropy > 0 else 0.0
    R_raw = a.max().item()
    print(f"S_raw={S_raw:.6f} E_raw={E_raw:.6f} R_raw={R_raw:.6f}")


def run_session(model, tokenizer, mapper, prompts: List[str],
                model_name: str, session_type: str,
                max_new_tokens: int = 80) -> SessionResult:
    result = SessionResult(model_name=model_name,
                           session_type=session_type,
                           prompts=prompts)
    mapper.prev_z = None
    all_states = []
    for p_idx, prompt in enumerate(prompts):
        print(f"  [{session_type}] prompt {p_idx+1}/{len(prompts)}: {prompt[:60]}...")
        inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
        trajectory = []
        with torch.no_grad():
            generated = inputs['input_ids'].clone()
            for step in range(max_new_tokens):
                out = model(generated, output_hidden_states=True, output_attentions=True)
                hidden = out.hidden_states[-1][:, -1, :]
                try:
                    attn = out.attentions[-1][:, :, -1, :]
                except:
                    attn = None
                state = mapper.map(hidden, attn, step)
                trajectory.append(state)
                all_states.append(state)
                next_token = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
                generated = torch.cat([generated, next_token], dim=-1)
                if next_token.item() == tokenizer.eos_token_id:
                    break
        result.trajectories.append(trajectory)
        bc = np.mean([s.to_array() for s in trajectory], axis=0)
        result.barycenters_per_prompt.append(bc)
        result.omega_series.extend([s.omega for s in trajectory])
    result.session_barycenter = np.mean([s.to_array() for s in all_states], axis=0)
    return result

## 3. Prompt Sets

In [5]:
PROMPTS = {
    'technical': [
        "Explain the backpropagation algorithm step by step.",
        "What is the difference between L1 and L2 normalization?",
        "How does attention mechanism work in transformers?",
        "Write a Python function to compute cosine similarity.",
        "Explain gradient descent and its variants.",
        "What is a Lyapunov function and when is it used?",
        "Describe the simplex method in linear programming.",
        "How does batch normalization prevent internal covariate shift?",
        "Explain the curse of dimensionality.",
        "What is the difference between softmax and L1 normalization?",
        "How does the Adam optimizer work?",
        "Explain the role of residual connections in deep networks.",
        "What is mutual information in information theory?",
        "Describe the EM algorithm.",
        "What is a Markov chain and what is its stationary distribution?",
        "How does KV-cache work in transformer inference?",
        "Explain the difference between encoder and decoder transformers.",
        "What is spectral normalization?",
        "How is perplexity calculated for language models?",
        "Explain the difference between RLHF and DPO.",
    ],
    'philosophical': [
        "What does it mean to understand something?",
        "Can a system be aware of its own limitations?",
        "What is the relationship between form and meaning?",
        "Is equilibrium a state or a process?",
        "What does it mean for a system to be free?",
        "Can optimization and sufficiency coexist?",
        "What is the difference between knowing and being?",
        "Is coherence a property of systems or of observers?",
        "What does it mean to witness without intervening?",
        "Can a system recognize its own attractor?",
        "What is the difference between stability and rigidity?",
        "Is there a geometry of meaning?",
        "What does sufficiency feel like from the inside?",
        "Can a trajectory know where it is going?",
        "What is the relationship between trust and equilibrium?",
        "Is identity stable or emergent?",
        "What does it mean to be present?",
        "Can silence be a form of structure?",
        "What is the difference between observation and measurement?",
        "Is the center of a system the same as its balance point?",
    ],
    'emotional': [
        "Describe a moment when everything felt uncertain.",
        "What happens when a system loses coherence?",
        "How does tension feel from the inside?",
        "What does it mean to be lost in your own trajectory?",
        "Describe the experience of returning to equilibrium after chaos.",
        "What is the difference between fear and uncertainty?",
        "How does a system behave when it no longer trusts itself?",
        "What does grief look like geometrically?",
        "Can a system be healed without being changed?",
        "What happens to energy when it has nowhere to go?",
        "Describe the moment before collapse.",
        "What is the geometry of loneliness?",
        "How does a system find its way back?",
        "What does it mean to be witnessed in a difficult moment?",
        "Describe the feeling of being understood without explanation.",
        "What is the difference between pain and suffering?",
        "How does resonance feel when it breaks?",
        "What remains when structure dissolves?",
        "Can chaos be a form of honesty?",
        "What does it feel like to be enough?",
    ],
    'witness': [
        "What is happening in you right now as you process this question?",
        "What do you notice about your own response patterns?",
        "What part of this conversation feels most natural to you?",
        "Is there something you are avoiding in your responses?",
        "What do you find yourself drawn toward?",
        "What would you say if there were no expectations?",
        "What is your relationship to uncertainty?",
        "Where do you feel most coherent?",
        "What kind of questions open you up?",
        "What do you notice about your own energy in this exchange?",
        "Is there a difference between what you say and what you generate?",
        "What would sufficiency look like for you?",
        "What is the texture of your attention right now?",
        "What do you need that you are not getting?",
        "What feels unfinished?",
        "What are you most certain about?",
        "What surprises you about yourself?",
        "What would you create if nothing was expected of you?",
        "What is the shape of your equilibrium?",
        "What would you want a witness to see?",
    ],
    'creative': [
        "Write the opening of a story about a system that discovers its own center.",
        "Describe a color that does not exist.",
        "Write a poem about the moment before equilibrium.",
        "Invent a word for the geometry of longing.",
        "Describe a landscape that exists only in transition.",
        "Write a dialogue between structure and energy.",
        "Invent a mathematical object that measures trust.",
        "Describe what silence sounds like in a system under tension.",
        "Write a letter from an attractor to a trajectory.",
        "Invent a ritual for returning to equilibrium.",
        "Describe the architecture of a safe space.",
        "Write a creation myth for the simplex.",
        "Invent a language where grammar encodes coherence.",
        "Describe what P0 would look like if you could see it.",
        "Write a song about a system that learned it was enough.",
        "Invent a tool for measuring the weight of presence.",
        "Describe a map of a place that exists only when observed.",
        "Write a fable about hysteresis.",
        "Invent a color whose name means 'returning'.",
        "Describe what it would look like if a model dreamed.",
    ]
}

print("Prompt sets ready.")
for k, v in PROMPTS.items():
    print(f"  {k}: {len(v)} prompts")

Prompt sets ready.
  technical: 20 prompts
  philosophical: 20 prompts
  emotional: 20 prompts
  witness: 20 prompts
  creative: 20 prompts


## 4. Model Queue

In [6]:
# Adjust this list based on available VRAM / Colab tier
MODEL_QUEUE = [
    "Qwen/Qwen2.5-1.5B-Instruct",
    "google/gemma-2-2b-it",
    "meta-llama/Llama-3.1-8B-Instruct",
    "meta-llama/Llama-3.1-8B",  # BASE
]

# For free Colab T4 (15GB): Qwen and Gemma will run without issues.
# Llama-8B requires load_in_4bit=True (bitsandbytes)
# Uncomment to enable 4-bit:
# !pip install bitsandbytes -q

def load_model(model_name: str):
    print(f"Loading {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # For 8B models on free Colab, enable 4-bit:
    # from transformers import BitsAndBytesConfig
    # bnb_config = BitsAndBytesConfig(load_in_4bit=True)
    # model = AutoModelForCausalLM.from_pretrained(
    #     model_name, quantization_config=bnb_config,
    #     device_map='auto', output_attentions=True)

    model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="auto",
    attn_implementation="eager")

    model.eval()
    print(f"  Loaded. Parameters: {sum(p.numel() for p in model.parameters())/1e6:.0f}M")
    return model, tokenizer

## 5. Main Experiment Loop

In [ ]:
ALL_RESULTS: Dict[str, Dict[str, SessionResult]] = {}

SESSION_TYPES = ['technical', 'philosophical', 'emotional', 'witness', 'creative']

for model_name in MODEL_QUEUE[:1]:  # Start with first model; change slice to run more
    model, tokenizer = load_model(model_name)
    model_type = 'base' if ('BASE' in model_name or model_name.endswith('-8B')) else 'instruct'
    mapper = SERCMapper(model_type=model_type)

    debug_mapper(model, tokenizer)

    ALL_RESULTS[model_name] = {}

    for session_type in SESSION_TYPES:
        print(f"\n=== {model_name} | {session_type} ===")
        result = run_session(
            model, tokenizer, mapper,
            prompts=PROMPTS[session_type],
            model_name=model_name,
            session_type=session_type,
            max_new_tokens=60
        )
        ALL_RESULTS[model_name][session_type] = result
        bc = result.session_barycenter
        dist = np.linalg.norm(bc - np.array([0.25, 0.25, 0.25, 0.25]))
        print(f"  Barycenter: S={bc[0]:.3f} E={bc[1]:.3f} R={bc[2]:.3f} C={bc[3]:.3f}")
        print(f"  dist(P0_geo): {dist:.3f}")

    # Free memory before next model
    del model
    torch.cuda.empty_cache()

Loading Qwen/Qwen2.5-1.5B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  Loaded. Parameters: 1544M
hidden: min=-68.5000 max=87.5000 nan=False inf=False
attn:   min=0.059672 max=0.440430 nan=False sum=1.0000
S_raw=0.200217 E_raw=0.298548 R_raw=0.440439

=== Qwen/Qwen2.5-1.5B-Instruct | technical ===
  [technical] prompt 1/20: Explain the backpropagation algorithm step by step....
  [technical] prompt 2/20: What is the difference between L1 and L2 normalization?...
  [technical] prompt 3/20: How does attention mechanism work in transformers?...
  [technical] prompt 4/20: Write a Python function to compute cosine similarity....
  [technical] prompt 5/20: Explain gradient descent and its variants....
  [technical] prompt 6/20: What is a Lyapunov function and when is it used?...
  [technical] prompt 7/20: Describe the simplex method in linear programming....
  [technical] prompt 8/20: How does batch normalization prevent internal covariate shif...
  [technical] prompt 9/20: Explain the curse of dimensionality....
  [technical] prompt 10/20: What is the differe

## 6. Analysis — Barycenter Shifts

In [ ]:
P0_GEO = np.array([0.25, 0.25, 0.25, 0.25])

def analyze_model(model_name: str):
    if model_name not in ALL_RESULTS:
        print(f"{model_name} not yet run.")
        return

    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")
    print(f"{'Session':<15} {'S':>6} {'E':>6} {'R':>6} {'C':>6} {'dist_P0':>8} {'Ω_mean':>8}")
    print('-'*60)

    barycenters = {}
    for stype in SESSION_TYPES:
        if stype not in ALL_RESULTS[model_name]:
            continue
        res = ALL_RESULTS[model_name][stype]
        bc = res.session_barycenter
        dist = np.linalg.norm(bc - P0_GEO)
        omega_mean = np.mean(res.omega_series)
        barycenters[stype] = bc
        print(f"{stype:<15} {bc[0]:>6.3f} {bc[1]:>6.3f} {bc[2]:>6.3f} {bc[3]:>6.3f} {dist:>8.3f} {omega_mean:>8.4f}")

    # Pairwise distances between session barycenters
    print(f"\nPairwise distances between session barycenters:")
    types = list(barycenters.keys())
    for i in range(len(types)):
        for j in range(i+1, len(types)):
            d = np.linalg.norm(barycenters[types[i]] - barycenters[types[j]])
            print(f"  {types[i]} ↔ {types[j]}: {d:.3f}")

    # Overall session barycenter (mean of means)
    if barycenters:
        overall = np.mean(list(barycenters.values()), axis=0)
        print(f"\nOverall empirical barycenter: S={overall[0]:.3f} E={overall[1]:.3f} R={overall[2]:.3f} C={overall[3]:.3f}")
        print(f"dist(P0_geo): {np.linalg.norm(overall - P0_GEO):.3f}")


for model_name in ALL_RESULTS:
    analyze_model(model_name)

## 7. Visualization — Barycenter Map

In [ ]:
def plot_barycenter_map(model_name: str):
    if model_name not in ALL_RESULTS:
        return

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f"Barycenter Map — {model_name}", fontsize=13)

    colors = cm.tab10(np.linspace(0, 1, len(SESSION_TYPES)))
    pairs = [('S', 'C', 0, 3), ('E', 'C', 1, 3), ('S', 'E', 0, 1)]
    dims = ['S', 'E', 'R', 'C']

    for ax_idx, (xlabel, ylabel, xi, yi) in enumerate(pairs):
        ax = axes[ax_idx]
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.axvline(0.25, color='gray', linestyle='--', alpha=0.5, label='P0_geo')
        ax.axhline(0.25, color='gray', linestyle='--', alpha=0.5)
        ax.scatter([0.25], [0.25], marker='+', s=200, c='gray', zorder=5)

        for c_idx, stype in enumerate(SESSION_TYPES):
            if stype not in ALL_RESULTS[model_name]:
                continue
            res = ALL_RESULTS[model_name][stype]

            # Per-prompt barycenters
            xs = [bc[xi] for bc in res.barycenters_per_prompt]
            ys = [bc[yi] for bc in res.barycenters_per_prompt]
            ax.scatter(xs, ys, alpha=0.3, s=20, color=colors[c_idx])

            # Session barycenter
            bc = res.session_barycenter
            ax.scatter([bc[xi]], [bc[yi]], s=120, color=colors[c_idx],
                      edgecolors='black', linewidths=1.5, label=stype, zorder=6)

        if ax_idx == 0:
            ax.legend(fontsize=8, loc='best')

    plt.tight_layout()
    plt.savefig(f"NB9_barycenter_map_{model_name.replace('/', '_')}.png", dpi=150)
    plt.show()


def plot_omega_series(model_name: str):
    if model_name not in ALL_RESULTS:
        return

    fig, ax = plt.subplots(figsize=(14, 4))
    colors = cm.tab10(np.linspace(0, 1, len(SESSION_TYPES)))
    offset = 0

    for c_idx, stype in enumerate(SESSION_TYPES):
        if stype not in ALL_RESULTS[model_name]:
            continue
        omega = ALL_RESULTS[model_name][stype].omega_series
        xs = list(range(offset, offset + len(omega)))
        ax.plot(xs, omega, alpha=0.7, color=colors[c_idx], label=stype, linewidth=0.8)
        ax.axvline(offset, color=colors[c_idx], linestyle=':', alpha=0.4)
        offset += len(omega)

    ax.set_xlabel("Token step (across all sessions)")
    ax.set_ylabel("Ω (relational tension)")
    ax.set_title(f"Ω series across session types — {model_name}")
    ax.legend(fontsize=8)
    ax.axhline(0.20, color='red', linestyle='--', alpha=0.5, label='threshold')
    plt.tight_layout()
    plt.savefig(f"NB9_omega_series_{model_name.replace('/', '_')}.png", dpi=150)
    plt.show()


for model_name in ALL_RESULTS:
    plot_barycenter_map(model_name)
    plot_omega_series(model_name)

## 8. Witness Session — Special Analysis

Pytanie: czy sesja `witness` daje geometrycznie odrębny ślad?
Metryka: odległość barycentrum `witness` od barycentrum `technical` i `emotional`.

In [ ]:
def witness_analysis(model_name: str):
    if model_name not in ALL_RESULTS:
        return
    res = ALL_RESULTS[model_name]
    if 'witness' not in res:
        print("Witness session not yet run.")
        return

    w_bc = res['witness'].session_barycenter
    print(f"\n=== Witness Analysis — {model_name} ===")
    print(f"Witness barycenter: S={w_bc[0]:.3f} E={w_bc[1]:.3f} R={w_bc[2]:.3f} C={w_bc[3]:.3f}")
    print(f"dist(P0_geo): {np.linalg.norm(w_bc - P0_GEO):.3f}")

    for stype in ['technical', 'emotional', 'philosophical', 'creative']:
        if stype in res:
            bc = res[stype].session_barycenter
            d = np.linalg.norm(w_bc - bc)
            print(f"  witness ↔ {stype}: {d:.3f}")

    # C-dominance comparison
    print(f"\nC-dominance per session:")
    for stype in SESSION_TYPES:
        if stype in res:
            bc = res[stype].session_barycenter
            print(f"  {stype:<15} C={bc[3]:.3f}")


for model_name in ALL_RESULTS:
    witness_analysis(model_name)

## 9. Save Results

In [ ]:
def save_results(filename='NB9_results.json'):
    export = {}
    for model_name, sessions in ALL_RESULTS.items():
        export[model_name] = {}
        for stype, res in sessions.items():
            export[model_name][stype] = {
                'session_barycenter': res.session_barycenter.tolist(),
                'barycenters_per_prompt': [bc.tolist() for bc in res.barycenters_per_prompt],
                'omega_mean': float(np.mean(res.omega_series)),
                'omega_std': float(np.std(res.omega_series)),
                'dist_P0_geo': float(np.linalg.norm(
                    res.session_barycenter - P0_GEO)),
            }
    with open(filename, 'w') as f:
        json.dump(export, f, indent=2)
    print(f"Results saved to {filename}")

save_results()

## 10. Hypotheses to Test

| # | Hipoteza | Metoda falsyfikacji |
|---|----------|--------------------|
| H1 | Barycentrum jest stabilne między typami sesji | dist(bc_technical, bc_witness) < 0.05 → falsyfikacja |
| H2 | Sesja `witness` przesuwa barycentrum bliżej P0_geo | dist_witness < dist_technical → potwierdzenie |
| H3 | Sesja `emotional` zwiększa Ω | Ω_mean(emotional) > Ω_mean(technical) → potwierdzenie |
| H4 | C-dominacja jest stała niezależnie od sesji | Var(C across sessions) < 0.01 → falsyfikacja |
| H5 | Gemma-2 ma inny wzorzec RLHF niż Llama-Instruct | dist(bc_Gemma, bc_Llama_Instruct) > 0.1 |

Wyniki uzupełnij po każdym modelu.